# DAY 17 — Controlled YOLOv8n Refinement

Tune learning rate, augmentation and key settings with one controlled trial. Reuse completed Day 16 work and never repeat completed training epochs.

In [1]:

# DAY 17.1 — Robust setup, paths and Dataset V1 verification
from pathlib import Path
import os, json, yaml, zipfile, shutil, hashlib, random, time
from collections import Counter, defaultdict
import pandas as pd
import numpy as np

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print("Drive mount:", e)

PROJECT_DIR = Path("/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs")
DATASET_ROOT = Path("/content/TrashCAN_YOLO_Dataset_V1")
DATA_YAML = DATASET_ROOT / "data.yaml"
SOURCE_ZIP = Path("/content/drive/MyDrive/INTERNSHIPWORK/dataset.zip")
SOURCE_ROOT = Path("/content/TrashCAN")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Day 15/16/17 run directory not found:\n{PROJECT_DIR}\n"
        "Expected: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs"
    )

# Dataset V1 is temporary Colab storage. If it is missing, rebuild it once
# from the same source ZIP used by Day 16. This does NOT retrain any model.
if not DATA_YAML.exists():
    print("Dataset V1 is missing in this runtime. Rebuilding data only (no training)...")
    if not SOURCE_ZIP.exists():
        raise FileNotFoundError(
            f"Source dataset ZIP not found:\n{SOURCE_ZIP}"
        )

    if not SOURCE_ROOT.exists():
        with zipfile.ZipFile(SOURCE_ZIP, "r") as z:
            z.extractall(SOURCE_ROOT)

    material_candidates = [
        SOURCE_ROOT / "dataset" / "material_version",
        SOURCE_ROOT / "material_version"
    ]
    material_path = next((p for p in material_candidates if p.exists()), None)
    if material_path is None:
        matches = list(SOURCE_ROOT.rglob("material_version"))
        material_path = matches[0] if matches else None
    if material_path is None:
        raise FileNotFoundError("material_version folder not found in source dataset.")

    train_path = material_path / "train"
    val_path = material_path / "val"
    train_json = material_path / "instances_train_trashcan.json"
    val_json = material_path / "instances_val_trashcan.json"

    for p in [train_path, val_path, train_json, val_json]:
        if not p.exists():
            raise FileNotFoundError(f"Required source item missing:\n{p}")

    with open(train_json, encoding="utf-8") as f:
        train_data = json.load(f)
    with open(val_json, encoding="utf-8") as f:
        val_data = json.load(f)

    categories = train_data["categories"]
    garbage_categories = sorted(
        [c for c in categories if c["name"].startswith("trash_")],
        key=lambda x: x["name"]
    )
    class_map = {c["id"]: i for i, c in enumerate(garbage_categories)}
    class_names = [c["name"] for c in garbage_categories]

    # Preserve the Day 16 split definition:
    # original train pool -> 80% train + 20% test
    # original validation pool -> retained as val.
    image_exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    records = []
    ann_lookup = defaultdict(list)

    for pool, folder, coco in [
        ("source_train", train_path, train_data),
        ("source_val", val_path, val_data)
    ]:
        for img in coco["images"]:
            filename = os.path.basename(str(img["file_name"]))
            image_path = folder / filename
            if not image_path.is_file():
                raise FileNotFoundError(f"Image missing: {image_path}")
            records.append({
                "source_pool": pool,
                "filename": filename,
                "image_path": str(image_path),
                "width": int(img["width"]),
                "height": int(img["height"]),
                "image_id": int(img["id"])
            })
        for ann in coco["annotations"]:
            ann_lookup[(pool, int(ann["image_id"]))].append(ann)

    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()

    for r in records:
        r["sha256"] = sha256_file(r["image_path"])

    df = pd.DataFrame(records)
    train_pool = df[df.source_pool == "source_train"].copy()
    val_pool = df[df.source_pool == "source_val"].copy()

    from sklearn.model_selection import GroupShuffleSplit
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    tr_idx, te_idx = next(gss.split(train_pool, groups=train_pool["sha256"]))

    train_df = train_pool.iloc[tr_idx].copy()
    test_df = train_pool.iloc[te_idx].copy()
    val_df = val_pool.copy()

    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"
    split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

    if DATASET_ROOT.exists():
        shutil.rmtree(DATASET_ROOT)

    for split in ["train", "val", "test"]:
        (DATASET_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
        (DATASET_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

    def coco_to_yolo(bbox, W, H):
        x, y, w, h = map(float, bbox)
        x1, y1 = max(0, min(x, W)), max(0, min(y, H))
        x2, y2 = max(0, min(x+w, W)), max(0, min(y+h, H))
        bw, bh = x2-x1, y2-y1
        if bw <= 0 or bh <= 0:
            return None
        vals = [(x1+x2)/2/W, (y1+y2)/2/H, bw/W, bh/H]
        return [min(1.0, max(0.0, v)) for v in vals]

    from shutil import copy2
    for _, r in split_df.iterrows():
        split = r["split"]
        src = Path(r["image_path"])
        dst_img = DATASET_ROOT / "images" / split / r["filename"]
        dst_lbl = DATASET_ROOT / "labels" / split / f"{Path(r['filename']).stem}.txt"
        copy2(src, dst_img)
        lines = []
        for ann in ann_lookup[(r["source_pool"], int(r["image_id"]))]:
            cid = int(ann["category_id"])
            if cid not in class_map:
                continue
            box = coco_to_yolo(ann["bbox"], int(r["width"]), int(r["height"]))
            if box is not None:
                lines.append(f"{class_map[cid]} " + " ".join(f"{v:.6f}" for v in box))
        dst_lbl.write_text("\n".join(lines), encoding="utf-8")

    data_cfg = {
        "path": str(DATASET_ROOT),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": class_names
    }
    with open(DATA_YAML, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_cfg, f, sort_keys=False)

    split_df.to_csv(DATASET_ROOT / "split_manifest.csv", index=False)
    print("Dataset V1 rebuilt successfully. No model training was repeated.")

with open(DATA_YAML, encoding="utf-8") as f:
    DATA_CONFIG = yaml.safe_load(f)

TRAIN_DIR = DATASET_ROOT / DATA_CONFIG["train"]
VAL_DIR = DATASET_ROOT / DATA_CONFIG["val"]
TEST_DIR = DATASET_ROOT / DATA_CONFIG["test"]
CLASS_NAMES = DATA_CONFIG["names"]

for p in [DATA_YAML, TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Required path missing: {p}")

print("="*70)
print("DAY 17 SETUP PASS")
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_YAML  :", DATA_YAML)
print("TRAIN_DIR  :", TRAIN_DIR)
print("VAL_DIR    :", VAL_DIR)
print("TEST_DIR   :", TEST_DIR)
print("Classes    :", CLASS_NAMES)
print("="*70)


Mounted at /content/drive
Dataset V1 is missing in this runtime. Rebuilding data only (no training)...
Dataset V1 rebuilt successfully. No model training was repeated.
DAY 17 SETUP PASS
PROJECT_DIR: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs
DATA_YAML  : /content/TrashCAN_YOLO_Dataset_V1/data.yaml
TRAIN_DIR  : /content/TrashCAN_YOLO_Dataset_V1/images/train
VAL_DIR    : /content/TrashCAN_YOLO_Dataset_V1/images/val
TEST_DIR   : /content/TrashCAN_YOLO_Dataset_V1/images/test
Classes    : ['trash_etc', 'trash_fabric', 'trash_fishing_gear', 'trash_metal', 'trash_paper', 'trash_plastic', 'trash_rubber', 'trash_wood']


In [2]:

# DAY 17.2 — Automatically locate the completed Day 16 checkpoint
import pandas as pd
from pathlib import Path

def completed_run_candidates(prefix):
    items = []
    for run_dir in sorted(PROJECT_DIR.glob(prefix + "*")):
        csv_file = run_dir / "results.csv"
        best_pt = run_dir / "weights" / "best.pt"
        last_pt = run_dir / "weights" / "last.pt"
        if not csv_file.exists() or not best_pt.exists():
            continue
        try:
            df = pd.read_csv(csv_file)
            df.columns = [c.strip() for c in df.columns]
            if "epoch" in df.columns and not df.empty:
                items.append({
                    "epoch": int(df["epoch"].max()),
                    "run_dir": run_dir,
                    "csv": csv_file,
                    "best_pt": best_pt,
                    "last_pt": last_pt
                })
        except Exception as e:
            print("Skipping", run_dir, ":", e)
    return sorted(items, key=lambda x: x["epoch"], reverse=True)

day16_runs = completed_run_candidates("day16_yolov8n_img832")
if not day16_runs:
    raise FileNotFoundError(
        f"No completed Day 16 run found in {PROJECT_DIR}.\n"
        "A Day 16 results.csv and weights/best.pt are required."
    )

DAY16_INFO = day16_runs[0]
DAY16_RUN_DIR = DAY16_INFO["run_dir"]
DAY16_BEST_PT = DAY16_INFO["best_pt"]
DAY16_CSV = DAY16_INFO["csv"]

r16 = pd.read_csv(DAY16_CSV)
r16.columns = [c.strip() for c in r16.columns]
best16_row = r16.loc[r16["metrics/mAP50-95(B)"].idxmax()]

print("Selected Day 16 run:", DAY16_RUN_DIR)
print("Completed epoch:", DAY16_INFO["epoch"])
print("Day 16 best.pt:", DAY16_BEST_PT)
print("Best mAP50:", round(float(best16_row["metrics/mAP50(B)"]), 5))
print("Best mAP50-95:", round(float(best16_row["metrics/mAP50-95(B)"]), 5))


Selected Day 16 run: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day16_yolov8n_img832-28
Completed epoch: 100
Day 16 best.pt: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day16_yolov8n_img832-28/weights/best.pt
Best mAP50: 0.59674
Best mAP50-95: 0.42982


In [3]:
# DAY 17.0 — Install Ultralytics into the CURRENT notebook environment

import sys
import subprocess

print("Python used by this notebook:")
print(sys.executable)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "ultralytics==8.4.155"
])

print("\nUltralytics installation completed.")

Python used by this notebook:
/usr/bin/python3

Ultralytics installation completed.


In [4]:
# DAY 17.2 — Verify installation

import sys
import importlib.util

print("Python:", sys.executable)

spec = importlib.util.find_spec("ultralytics")

if spec is None:
    raise ModuleNotFoundError(
        "Ultralytics is STILL not available in this notebook environment."
    )

import ultralytics
import torch

print("\nUltralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available.")

Python: /usr/bin/python3
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

Ultralytics: 8.4.155
PyTorch: 2.11.0+cpu
CUDA available: False


In [5]:
from pathlib import Path
import pandas as pd
import os

PROJECT_DIR = Path("/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs")

# If Drive is not mounted, mount it
if not PROJECT_DIR.exists():
    from google.colab import drive
    drive.mount("/content/drive")

print("PROJECT_DIR:", PROJECT_DIR)
print("Exists:", PROJECT_DIR.exists())

# Find ALL Day 17 runs
day17_candidates = []

for run_dir in PROJECT_DIR.glob("day17_yolov8n_tuned_lr_aug*"):
    csv_file = run_dir / "results.csv"
    last_pt = run_dir / "weights" / "last.pt"
    best_pt = run_dir / "weights" / "best.pt"

    if csv_file.exists() and last_pt.exists():
        try:
            df = pd.read_csv(csv_file)
            df.columns = [c.strip() for c in df.columns]

            if "epoch" in df.columns and not df.empty:
                last_epoch = int(df["epoch"].max())

                day17_candidates.append(
                    (last_epoch, run_dir, csv_file, last_pt, best_pt)
                )
        except Exception as e:
            print("Could not read:", run_dir, e)

if not day17_candidates:
    raise FileNotFoundError(
        "No Day 17 run with last.pt and results.csv was found."
    )

# Select the run that has progressed the furthest
day17_candidates.sort(key=lambda x: x[0], reverse=True)

DAY17_EPOCH, DAY17_RUN_DIR, DAY17_CSV, DAY17_LAST, DAY17_BEST = day17_candidates[0]

print("\nSelected Day 17 run:")
print("Run folder :", DAY17_RUN_DIR)
print("Last epoch :", DAY17_EPOCH)
print("last.pt    :", DAY17_LAST)
print("best.pt    :", DAY17_BEST)

PROJECT_DIR: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs
Exists: True

Selected Day 17 run:
Run folder : /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_lr_aug-4
Last epoch : 2
last.pt    : /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_lr_aug-4/weights/last.pt
best.pt    : /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_lr_aug-4/weights/best.pt


In [6]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs")

candidates = []

for run_dir in PROJECT_DIR.glob("day17_yolov8n_tuned_lr_aug*"):
    csv_file = run_dir / "results.csv"
    last_pt = run_dir / "weights" / "last.pt"

    if csv_file.exists() and last_pt.exists():
        try:
            df = pd.read_csv(csv_file)
            df.columns = [c.strip() for c in df.columns]

            if "epoch" in df.columns and len(df) > 0:
                last_epoch = int(df["epoch"].max())
                candidates.append(
                    (last_epoch, run_dir, csv_file, last_pt)
                )
        except Exception as e:
            print("Skipping:", run_dir, e)

if not candidates:
    raise FileNotFoundError("Day 17 last.pt not found.")

candidates.sort(key=lambda x: x[0], reverse=True)

EXISTING_EPOCH, DAY17_RUN_DIR, DAY17_CSV, DAY17_LAST = candidates[0]

print("Selected Day 17 run:")
print(DAY17_RUN_DIR)
print()
print("Last completed epoch:", EXISTING_EPOCH)
print("Checkpoint:", DAY17_LAST)

Selected Day 17 run:
/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_lr_aug-4

Last completed epoch: 2
Checkpoint: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_lr_aug-4/weights/last.pt


In [7]:
from ultralytics import YOLO

model_day17 = YOLO(str(DAY17_LAST))

print("Loaded Day 17 checkpoint successfully:")
print(DAY17_LAST)

Loaded Day 17 checkpoint successfully:
/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_lr_aug-4/weights/last.pt


In [8]:
remaining_epochs = 100 - (EXISTING_EPOCH + 1)

print("Existing completed epoch:", EXISTING_EPOCH)
print("Remaining epochs:", remaining_epochs)

Existing completed epoch: 2
Remaining epochs: 97


In [9]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs")

rows = []

for run_dir in sorted(PROJECT_DIR.glob("day17_yolov8n_*")):
    csv_file = run_dir / "results.csv"
    last_pt = run_dir / "weights" / "last.pt"
    best_pt = run_dir / "weights" / "best.pt"

    if csv_file.exists():
        try:
            df = pd.read_csv(csv_file)
            df.columns = [c.strip() for c in df.columns]

            if "epoch" in df.columns and len(df) > 0:
                max_epoch = int(df["epoch"].max()) + 1

                rows.append({
                    "Run": run_dir.name,
                    "Epochs saved": max_epoch,
                    "last.pt": last_pt.exists(),
                    "best.pt": best_pt.exists()
                })

        except Exception as e:
            print("Error reading:", run_dir.name, e)

result = pd.DataFrame(rows)

if len(result) > 0:
    result = result.sort_values(
        by="Epochs saved",
        ascending=False
    ).reset_index(drop=True)

display(result)

,Run,Epochs saved,last.pt,best.pt
0,day17_yolov8n_tuned_continuation,97,True,True
1,day17_yolov8n_tuned_lr_aug-4,3,True,True
2,day17_yolov8n_tuned_lr_aug-5,2,True,True


In [10]:
from ultralytics import YOLO
from pathlib import Path

DAY17_RUN_DIR = Path(
    "/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_continuation"
)

DAY17_LAST = DAY17_RUN_DIR / "weights" / "last.pt"

print("Checkpoint:")
print(DAY17_LAST)

model_day17 = YOLO(str(DAY17_LAST))

Checkpoint:
/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_continuation/weights/last.pt


In [12]:
model_day17.train(
    resume=True,
    amp=False
)

New https://pypi.org/project/ultralytics/8.4.163 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/TrashCAN_YOLO_Dataset_V1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=97, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.022, hsv_s=0.12, hsv_v=0.08, imgsz=832, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=t

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7abbad11fed0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

In [19]:
# DAY 17.4 — Overfitting, class imbalance, best validation config and refinement log
from collections import Counter
import json, numpy as np, pandas as pd
import yaml # Import yaml for loading the config file

r17 = pd.read_csv(DAY17_CSV)
r17.columns = [c.strip() for c in r17.columns]

DAY17_RUN_NAME = DAY17_RUN_DIR.name

# Load the training configuration from args.yaml in the run directory
with open(DAY17_RUN_DIR / "args.yaml", "r") as f:
    DAY17_CONFIG = yaml.safe_load(f)

# Best validation epoch is defined consistently as the epoch with highest mAP50-95.
best_idx = r17["metrics/mAP50-95(B)"].idxmax()
best17 = r17.loc[best_idx]
last17 = r17.iloc[-1]

p = float(best17["metrics/precision(B)"])
r = float(best17["metrics/recall(B)"])
f1 = 2*p*r/(p+r) if (p+r) else 0.0
# Descriptive overfitting signal: compare best validation mAP50-95 with final value.
m_best = float(best17["metrics/mAP50-95(B)"])
m_last = float(last17["metrics/mAP50-95(B)"])
m_gap = m_best - m_last

if m_gap > 0.03:
    overfit_note = "Validation mAP50-95 ended noticeably below its best recorded value; monitor for overfitting."
elif m_gap > 0.01:
    overfit_note = "A small best-to-final validation gap is present; monitor the final training stage."
else:
  overfit_note = "No large best-to-final validation mAP50-95 gap was observed."

# Training class counts from actual Dataset V1 labels.
class_counts = Counter()
for label_file in sorted((DATASET_ROOT / "labels" / "train").glob("*.txt")):
    text = label_file.read_text(encoding="utf-8").strip()
    for line in text.splitlines():
        if line.strip():
            class_counts[int(line.split()[0])] += 1

balance_rows = []
for i, name in enumerate(CLASS_NAMES):
    balance_rows.append({
        "class_id": i,
        "class": name,
        "train_objects": class_counts.get(i, 0)
    })
balance_df = pd.DataFrame(balance_rows)
balance_df["percentage"] = (
    balance_df["train_objects"] / max(1, balance_df["train_objects"].sum()) * 100
)
max_count = int(balance_df["train_objects"].max())
min_positive = int(balance_df.loc[balance_df["train_objects"] > 0, "train_objects"].min())
imbalance_ratio = max_count / min_positive if min_positive else float("inf")

display(balance_df.sort_values("train_objects", ascending=False).reset_index(drop=True))

refinement_log = {
    "day": 17,
    "experiment": DAY17_RUN_NAME,
    "starting_checkpoint": str(DAY16_BEST_PT),
    "configuration": DAY17_CONFIG,
    "training_completed_epoch": int(r17["epoch"].max()),
    "best_validation_epoch": int(best17["epoch"]),
    "best_validation_metrics": {
        "mAP50": float(best17["metrics/mAP50(B)"]),
        "mAP50_95": float(best17["metrics/mAP50-95(B)"]),
        "precision": p,
        "recall": r,
        "F1": f1
    },
    "overfitting_check": {
        "best_mAP50_95": m_best,
        "final_mAP50_95": m_last,
        "best_minus_final_gap": m_gap,
        "note": overfit_note
    },
    "class_imbalance": {
        "max_min_positive_object_ratio": imbalance_ratio,
        "counts": {str(k): int(v) for k, v in class_counts.items()},
        "note": "Class-wise validation metrics are used to monitor rare classes; no excessive hyperparameter search was performed."
    }
}

REFINEMENT_LOG = DAY17_RUN_DIR / "refinement_log.json"
with open(REFINEMENT_LOG, "w", encoding="utf-8") as f:
    json.dump(refinement_log, f, indent=2)

balance_df.to_csv(DAY17_RUN_DIR / "class_balance_train.csv", index=False)

print("="*70)
print("DAY 17 COMPLETE")
print("Best epoch:", int(best17["epoch"]))
print("Best mAP50:", round(float(best17["metrics/mAP50(B)"]), 5))
print("Best mAP50-95:", round(float(best17["metrics/mAP50-95(B)"]), 5))
print("Precision:", round(p, 5))
print("Recall:", round(r, 5))
print("F1:", round(f1, 5))
print("Refinement log:", REFINEMENT_LOG)
print("Tuned detector:", DAY17_BEST)
print("Overfitting:", overfit_note)
print("="*70)

,class_id,class,train_objects,percentage
0,0,trash_etc,1303,33.012414
1,5,trash_plastic,1181,29.921459
2,3,trash_metal,718,18.191031
3,7,trash_wood,215,5.447175
4,1,trash_fabric,197,4.991133
5,4,trash_paper,133,3.369648
6,2,trash_fishing_gear,110,2.786927
7,6,trash_rubber,90,2.280213


DAY 17 COMPLETE
Best epoch: 2
Best mAP50: 0.03707
Best mAP50-95: 0.01326
Precision: 0.33768
Recall: 0.0623
F1: 0.10519
Refinement log: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_continuation/refinement_log.json
Tuned detector: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_lr_aug-4/weights/best.pt
Overfitting: No large best-to-final validation mAP50-95 gap was observed.
